In [ ]:
import searvey
import pandas as pd
import hvplot.pandas
import ioc_cleanup as C

In [ ]:
ioc_all = C.get_meta()
ioc_all

In [ ]:
ioc_all.hvplot.points(x="lon", y="lat", tiles=True, geo=True, c='r', hover_cols='ioc_code', s=50, line_color='k').opts(width=800, height=700)

In [ ]:
!mkdir -p data

In [ ]:
YEAR = 2011

In [ ]:
import os
def download_year_station(ioc_codes: list[str], year: int, data_folder: str = "./data"):
    year_folder = f"{data_folder}/{year}"
    os.makedirs(year_folder, exist_ok=True)
    try:
        dict_df = C.download_raw(ioc_codes, f"{year}-01-01", f"{year}-12-31T23:59:59")
        for station in ioc_codes:
            df = dict_df[station]
            if not df.empty:
                df.to_parquet(f"{year_folder}/{station}.parquet")
                print(f"  Saved {station} for {year}")
    except Exception as e:
        print(f"  Error for {station} in {year}: {e}")

# download_year_station(ioc_all.ioc_code.tolist(), YEAR, data_folder="../data")

In [ ]:
# for i_s, station in ioc_all.iterrows():
#     df = C.load_station(station.ioc_code, data_dir="../data" )
#     if not df.empty:
#         for sensor in df.columns:
#             df[[sensor]].to_parquet(f"../raw/{station.ioc_code}_{sensor}.parquet")

In [ ]:
import re
import glob
pattern = r"raw/([\w\d]+)\.parquet"
stations_sensors = [re.search(pattern, path).group(1) for path in glob.glob("../raw/*parquet")]
f"Total decoupled stations: {len(stations_sensors)}"

In [ ]:
import pathlib
all_stats = C._statistics.calc_statistics(ioc_all, pathlib.Path("../raw"))

In [ ]:
all_stats[all_stats.detide_ratio > 1].detide_ratio = 1
all_stats = all_stats[all_stats.detide_ratio > 0.8]

In [ ]:
all_stats

resize and write data only for June-October 2011

In [ ]:
!mkdir -p ../validation_2011

In [ ]:
station = "ajac2"
for station in all_stats.ioc_code.tolist():
    obsfile = glob.glob(f"../raw/{station}*.parquet")[0]
    obs = pd.read_parquet(obsfile)
    obs = obs.sort_index()
    obs = obs.dropna()
    obs= obs.loc["2011-06-01":"2011-10"]
    if not obs.empty:
        obs = obs[abs(obs-obs.mean())<obs.std()*3]
        # plot = obs.hvplot(c="k", line_dash="dashed")
        # plot.opts(width=1200, height=800, title=station)
        obs.to_parquet(f"../validation_2011/{obsfile.split('raw/')[1]}")

In [ ]:
pattern = r"validation_2011/([\w\d]+)\.parquet"
stations_sensors = [re.search(pattern, path).group(1) for path in glob.glob("../validation_2011/*parquet")]
f"Total decoupled stations: {len(stations_sensors)}"

In [ ]:
discard_stations = ['rodr','mini','coch','chenn','vish','krna','chtt','moul','nanc','ganm','lamu','momb','pemba','toul','newl','mali','sona','beno','cila', 
    'dkam','dsen','vodo','drwp','quepo','cpla','dmon','const','dnwp','vodo', "syow", "figu2", "rous2", "sole2", "dhaw", "seve2", "rous"
    # "cili",
]

In [ ]:
stations = []
for ss in stations_sensors:
    station, sensor = ss.split('_')
    stations.append(station)
    obsfile = f"../validation_2011/{ss}.parquet"
    obs = pd.read_parquet(obsfile)
    s = ioc_all[ioc_all.ioc_code == station]
    obs.attrs['lon'] = s.lon.values[0]
    obs.attrs['lat'] = s.lat.values[0]
    obs.to_parquet(obsfile)

subset_2011 = ioc_all[ioc_all.ioc_code.isin(stations)]
subset_2011_good = subset_2011[~subset_2011.ioc_code.isin(discard_stations)]

subset_2011_good.hvplot.points(x="lon", y="lat", tiles=True, geo=True, c='r', hover_cols='ioc_code', s=50, line_color='k').opts(width=800, height=700)

In [ ]:
subset_2011_good.to_csv("ioc_validation_2011.csv")